In [2]:
# ============================================================
# STEP 1: CONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

print("Google Drive connected successfully.")

Mounted at /content/drive
Google Drive connected successfully.


In [3]:
# ============================================================
# STEP 2: DEFINE PROJECT PATHS
# ============================================================

import os
import pandas as pd
import numpy as np

# Main project folder
PROJECT_ROOT = "/content/drive/MyDrive/NNDL_English_Hindi_Translator"

# Raw dataset folder
RAW_DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "raw"
)

# Processed dataset folder
PROCESSED_DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

# Training dataset
TRAIN_PATH = os.path.join(
    RAW_DATA_DIR,
    "train-00000-of-00001.parquet"
)

print("Training file:")
print(TRAIN_PATH)

print("\nProcessed data will be saved in:")
print(PROCESSED_DATA_DIR)

Training file:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/data/raw/train-00000-of-00001.parquet

Processed data will be saved in:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/data/processed


In [4]:
# ============================================================
# STEP 3: LOAD RAW TRAINING DATA
# ============================================================

train_df = pd.read_parquet(
    TRAIN_PATH,
    columns=["translation"]
)

print("Raw training data loaded successfully.")
print(f"Original rows: {len(train_df):,}")

Raw training data loaded successfully.
Original rows: 1,659,083


In [5]:
# ============================================================
# STEP 4: EXTRACT ENGLISH AND HINDI TEXT
# ============================================================

train_df["english"] = train_df["translation"].apply(
    lambda x: x["en"] if x is not None else None
)

train_df["hindi"] = train_df["translation"].apply(
    lambda x: x["hi"] if x is not None else None
)

# Keep only the two columns we actually need.
train_df = train_df[
    ["english", "hindi"]
].copy()

print("English and Hindi columns extracted.")

display(train_df.head())

English and Hindi columns extracted.


,english,hindi
0,Give your application an accessibility workout,अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
1,Accerciser Accessibility Explorer,एक्सेर्साइसर पहुंचनीयता अन्वेषक
2,The default plugin layout for the bottom panel,निचले पटल के लिए डिफोल्ट प्लग-इन खाका
3,The default plugin layout for the top panel,ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका
4,A list of plugins that are disabled by default,उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...


In [6]:
# ============================================================
# STEP 5: STORE ORIGINAL DATASET SIZE
# ============================================================

original_rows = len(train_df)

print(f"Original training pairs: {original_rows:,}")

Original training pairs: 1,659,083


In [7]:
# ============================================================
# STEP 6: NORMALIZE WHITESPACE
# ============================================================

# Remove leading/trailing spaces.
train_df["english"] = train_df["english"].fillna("").str.strip()
train_df["hindi"] = train_df["hindi"].fillna("").str.strip()

# Replace repeated spaces/tabs/newlines with one normal space.
train_df["english"] = train_df["english"].str.replace(
    r"\s+",
    " ",
    regex=True
)

train_df["hindi"] = train_df["hindi"].str.replace(
    r"\s+",
    " ",
    regex=True
)

print("Whitespace normalized successfully.")

Whitespace normalized successfully.


In [8]:
# ============================================================
# STEP 7: REMOVE EMPTY SENTENCES
# ============================================================

before_empty_removal = len(train_df)

train_df = train_df[
    (train_df["english"] != "") &
    (train_df["hindi"] != "")
].copy()

after_empty_removal = len(train_df)

removed_empty = (
    before_empty_removal -
    after_empty_removal
)

print(f"Rows before empty removal : {before_empty_removal:,}")
print(f"Rows after empty removal  : {after_empty_removal:,}")
print(f"Rows removed              : {removed_empty:,}")

Rows before empty removal : 1,659,083
Rows after empty removal  : 1,652,981
Rows removed              : 6,102


In [9]:
# ============================================================
# STEP 8: REMOVE EXACT DUPLICATE TRANSLATION PAIRS
# ============================================================

before_duplicate_removal = len(train_df)

train_df = train_df.drop_duplicates(
    subset=["english", "hindi"]
).copy()

after_duplicate_removal = len(train_df)

removed_duplicates = (
    before_duplicate_removal -
    after_duplicate_removal
)

print(f"Rows before duplicate removal : {before_duplicate_removal:,}")
print(f"Rows after duplicate removal  : {after_duplicate_removal:,}")
print(f"Duplicates removed            : {removed_duplicates:,}")

Rows before duplicate removal : 1,652,981
Rows after duplicate removal  : 1,412,544
Duplicates removed            : 240,437


In [10]:
# ============================================================
# STEP 9: CALCULATE WORD COUNTS
# ============================================================

train_df["english_word_count"] = (
    train_df["english"]
    .str.split()
    .str.len()
)

train_df["hindi_word_count"] = (
    train_df["hindi"]
    .str.split()
    .str.len()
)

print("Word counts calculated.")

display(
    train_df[
        [
            "english",
            "english_word_count",
            "hindi",
            "hindi_word_count"
        ]
    ].head()
)

Word counts calculated.


,english,english_word_count,hindi,hindi_word_count
0,Give your application an accessibility workout,6,अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें,8
1,Accerciser Accessibility Explorer,3,एक्सेर्साइसर पहुंचनीयता अन्वेषक,3
2,The default plugin layout for the bottom panel,8,निचले पटल के लिए डिफोल्ट प्लग-इन खाका,7
3,The default plugin layout for the top panel,8,ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका,7
4,A list of plugins that are disabled by default,9,उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...,12


In [11]:
# ============================================================
# STEP 10: FILTER SENTENCES BY LENGTH
# ============================================================

MIN_WORDS = 1
MAX_WORDS = 50

before_length_filter = len(train_df)

train_df = train_df[
    train_df["english_word_count"].between(
        MIN_WORDS,
        MAX_WORDS
    )
    &
    train_df["hindi_word_count"].between(
        MIN_WORDS,
        MAX_WORDS
    )
].copy()

after_length_filter = len(train_df)

removed_by_length = (
    before_length_filter -
    after_length_filter
)

print(f"Length rule: {MIN_WORDS} to {MAX_WORDS} words")
print()
print(f"Rows before length filtering : {before_length_filter:,}")
print(f"Rows after length filtering  : {after_length_filter:,}")
print(f"Rows removed by length       : {removed_by_length:,}")

Length rule: 1 to 50 words

Rows before length filtering : 1,412,544
Rows after length filtering  : 1,351,975
Rows removed by length       : 60,569


In [12]:
# ============================================================
# STEP 11: VERIFY CLEANED DATA
# ============================================================

print("CLEANED DATASET CHECK")
print("-" * 45)

print(f"Rows                     : {len(train_df):,}")

print(
    "Missing English            :",
    train_df["english"].isnull().sum()
)

print(
    "Missing Hindi              :",
    train_df["hindi"].isnull().sum()
)

print(
    "Empty English              :",
    train_df["english"].eq("").sum()
)

print(
    "Empty Hindi                :",
    train_df["hindi"].eq("").sum()
)

print(
    "Exact duplicate pairs      :",
    train_df.duplicated(
        subset=["english", "hindi"]
    ).sum()
)

print(
    "Maximum English word count :",
    train_df["english_word_count"].max()
)

print(
    "Maximum Hindi word count   :",
    train_df["hindi_word_count"].max()
)

CLEANED DATASET CHECK
---------------------------------------------
Rows                     : 1,351,975
Missing English            : 0
Missing Hindi              : 0
Empty English              : 0
Empty Hindi                : 0
Exact duplicate pairs      : 0
Maximum English word count : 50
Maximum Hindi word count   : 50


In [14]:
# ============================================================
# STEP 12: CREATE FIXED 300,000-PAIR TRAINING SUBSET
# ============================================================

# Number of training pairs we will use for model training.
TRAIN_SUBSET_SIZE = 300_000

# Fixed random seed for reproducibility.
RANDOM_SEED = 42

# Randomly sample exactly 300,000 pairs
# from the cleaned training dataset.
train_300k = train_df.sample(
    n=TRAIN_SUBSET_SIZE,
    random_state=RANDOM_SEED
).copy()

# Reset row numbers so they run cleanly from 0 to 299,999.
train_300k = train_300k.reset_index(drop=True)

print("300K training subset created successfully.")
print(f"Subset rows: {len(train_300k):,}")

300K training subset created successfully.
Subset rows: 300,000


In [15]:
# ============================================================
# STEP 13: VERIFY TRAINING SUBSET
# ============================================================

print("300K TRAINING SUBSET CHECK")
print("-" * 45)

print(f"Rows                  : {len(train_300k):,}")

print(
    "Missing English         :",
    train_300k["english"].isnull().sum()
)

print(
    "Missing Hindi           :",
    train_300k["hindi"].isnull().sum()
)

print(
    "Duplicate pairs         :",
    train_300k.duplicated(
        subset=["english", "hindi"]
    ).sum()
)

print(
    "Maximum English words   :",
    train_300k["english_word_count"].max()
)

print(
    "Maximum Hindi words     :",
    train_300k["hindi_word_count"].max()
)

300K TRAINING SUBSET CHECK
---------------------------------------------
Rows                  : 300,000
Missing English         : 0
Missing Hindi           : 0
Duplicate pairs         : 0
Maximum English words   : 50
Maximum Hindi words     : 50


In [ ]:
# ============================================================
# STEP 14: PREPARE FINAL TRAINING FILE
# ============================================================

train_300k_final = train_300k[
    ["english", "hindi"]
].copy()

display(train_300k_final.head())

print("Final columns:")
print(train_300k_final.columns.tolist())

,english,hindi
0,The judgment of the Bombay High Court referred...,"मुंबई उच्च न्यायालय का निर्णय, जो ऊपर अनुच्छेद..."
1,I am sure you all will do well in life.,मुझे विश्वास है कि आप सभी जीवन में अच्छी तरक्क...
2,"Balconies, cupolas and towers surmount the pal...","बालकनी, कूपोला और बड़ी बड़ी मीनारें इस महल को ..."
3,All the temples of Kerala invariably turn out ...,"केरल में सब मंदिर चाहे वह विष्णु मंदिर हों, शि..."
4,President of India Shri Pranab Mukherjee parti...,भारत के राष्ट्रपति श्री प्रणब मुखर्जी आज (30 न...


Final columns:
['english', 'hindi']


In [ ]:
# ============================================================
# STEP 15: SAVE FINAL 300K TRAINING DATASET
# ============================================================

TRAIN_300K_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "train_clean_300k.parquet"
)

train_300k_final.to_parquet(
    TRAIN_300K_PATH,
    index=False
)

print("300K cleaned training dataset saved successfully.")
print("Saved to:")
print(TRAIN_300K_PATH)

300K cleaned training dataset saved successfully.
Saved to:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/data/processed/train_clean_300k.parquet


In [16]:
# ============================================================
# STEP 16: CHECK AVAILABLE DATAFRAMES
# ============================================================

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        print(name, value.shape)

train_df (1351975, 4)
train_300k (300000, 4)


In [17]:
# ============================================================
# STEP 17: CREATE 1,000,000-PAIR TRAINING SUBSET
# ============================================================
# We keep the existing 300K dataset unchanged.
# This larger subset will be used for the stronger Transformer.
# ============================================================

LARGE_TRAIN_SIZE = 1_000_000
RANDOM_SEED = 42

# Create reproducible 1M sample
train_1m = train_df.sample(
    n=LARGE_TRAIN_SIZE,
    random_state=RANDOM_SEED
).reset_index(
    drop=True
)

print(
    "1M training subset created successfully."
)

print(
    "Shape:",
    train_1m.shape
)

1M training subset created successfully.
Shape: (1000000, 4)


In [18]:
# ============================================================
# STEP 18: SAVE 1M TRAINING DATASET
# ============================================================

TRAIN_1M_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "train_clean_1m.parquet"
)

train_1m.to_parquet(
    TRAIN_1M_PATH,
    index=False
)

print("1M cleaned training dataset saved successfully.")
print("Saved to:")
print(TRAIN_1M_PATH)

1M cleaned training dataset saved successfully.
Saved to:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/data/processed/train_clean_1m.parquet


In [ ]:
# ============================================================
# STEP 16: LOAD OFFICIAL VALIDATION AND TEST DATA
# ============================================================

VALIDATION_PATH = os.path.join(
    RAW_DATA_DIR,
    "validation-00000-of-00001.parquet"
)

TEST_PATH = os.path.join(
    RAW_DATA_DIR,
    "test-00000-of-00001.parquet"
)

validation_df = pd.read_parquet(
    VALIDATION_PATH,
    columns=["translation"]
)

test_df = pd.read_parquet(
    TEST_PATH,
    columns=["translation"]
)

print(f"Validation rows: {len(validation_df):,}")
print(f"Test rows      : {len(test_df):,}")

Validation rows: 520
Test rows      : 2,507


In [ ]:
# ============================================================
# STEP 17: PREPARE VALIDATION AND TEST COLUMNS
# ============================================================

def extract_translation_columns(df):
    """
    Convert the nested 'translation' column into
    separate English and Hindi columns.
    """

    result = pd.DataFrame()

    result["english"] = df["translation"].apply(
        lambda x: x["en"] if x is not None else None
    )

    result["hindi"] = df["translation"].apply(
        lambda x: x["hi"] if x is not None else None
    )

    return result


validation_clean = extract_translation_columns(
    validation_df
)

test_clean = extract_translation_columns(
    test_df
)

print("Validation:")
display(validation_clean.head())

print("Test:")
display(test_clean.head())

Validation:


,english,hindi
0,Students of the Dattatreya city Municipal corp...,महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स...
1,With encouragement from Principal Sandhya Medp...,प्रधानाध्यापक संध्या मेडपल्लीवार के प्रोत्साहि...
2,"Rajesh Gavre, the President of the MNPA teache...",मनपा शिक्षक संघ के अध्यक्ष राजेश गवरे ने स्कूल...
3,Ramesh Saatpute examined the fort.,किले का परीक्षण रमेश सातपुते ने किया।
4,"Students like Nikhil Kavle, Darshan Gedekar, S...","किला निर्माण में निखिल कावले, दर्शन गेड़ेकर, स..."


Test:


,english,hindi
0,A black box in your car?,आपकी कार में ब्लैक बॉक्स?
1,As America's road planners struggle to find th...,"जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए..."
2,"The devices, which track every mile a motorist...","यह डिवाइस, जो मोटर-चालक द्वारा वाहन चलाए गए प्..."
3,The usually dull arena of highway planning has...,आम तौर पर हाईवे नियोजन जैसा उबाऊ काम भी अचानक ...
4,Libertarians have joined environmental groups ...,"आपने द्वारा ड्राइव किए गए मील, तथा संभवतः ड्रा..."


In [ ]:
# ============================================================
# STEP 18: NORMALIZE VALIDATION AND TEST TEXT
# ============================================================

for df in [validation_clean, test_clean]:

    # Replace missing values with empty strings
    df["english"] = df["english"].fillna("").str.strip()
    df["hindi"] = df["hindi"].fillna("").str.strip()

    # Normalize repeated whitespace
    df["english"] = df["english"].str.replace(
        r"\s+",
        " ",
        regex=True
    )

    df["hindi"] = df["hindi"].str.replace(
        r"\s+",
        " ",
        regex=True
    )

print("Validation and test text normalized.")

Validation and test text normalized.


In [ ]:
# ============================================================
# STEP 19: SAVE PROCESSED VALIDATION AND TEST FILES
# ============================================================

VALIDATION_CLEAN_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "validation_clean.parquet"
)

TEST_CLEAN_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "test_clean.parquet"
)

validation_clean.to_parquet(
    VALIDATION_CLEAN_PATH,
    index=False
)

test_clean.to_parquet(
    TEST_CLEAN_PATH,
    index=False
)

print("Processed validation and test files saved.")

print("\nValidation:")
print(VALIDATION_CLEAN_PATH)

print("\nTest:")
print(TEST_CLEAN_PATH)

Processed validation and test files saved.

Validation:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/data/processed/validation_clean.parquet

Test:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/data/processed/test_clean.parquet
